# 切粉検出・自律清掃デモ（SO-101 × YOLO-seg × MuJoCo）

加工後の作業台に残った切粉を、俯瞰カメラ画像から **YOLOセグメンテーション** で検出し、SO-101アームに付けたブラシで右側の回収トレーへ掃き寄せます。

上から順に実行すると、次の流れで最後にデモ動画（等速・1〜3分）が出力されます。

1. 環境準備（GPU確認・ライブラリ導入・SO-101モデル取得）
2. 動作確認（真値マスクで短い清掃）
3. 合成データセット生成
4. YOLO学習・評価
5. YOLOで清掃デモ → 動画出力

**ランタイム：** 「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選んでください。

## 1. 環境準備

In [ ]:
!nvidia-smi -L

リポジトリを取得します。`REPO_URL` を自分のGitHubリポジトリに書き換えてください。
（GitHubに上げる前に試す場合は、zipをアップロードして展開するセルを使ってください）

In [ ]:
REPO_URL = "https://github.com/<YOUR_ACCOUNT>/chip-cleaning-so101.git"  
!git clone -q {REPO_URL} /content/chip-cleaning-so101
%cd /content/chip-cleaning-so101

In [ ]:
# （代替）zip をアップロードして使う場合はこちらを実行
# from google.colab import files
# up = files.upload()            # chip-cleaning-so101.zip を選択
# !unzip -q -o chip-cleaning-so101.zip -d /content
# %cd /content/chip-cleaning-so101

In [ ]:
!pip install -q mujoco ultralytics "imageio[ffmpeg]" pyyaml opencv-python-headless

MuJoCo の描画をGPU（EGL）で行う設定です。

In [ ]:
import os
NVIDIA_ICD = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD):
    os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
    with open(NVIDIA_ICD, "w") as f:
        f.write('{"file_format_version" : "1.0.0", "ICD" : {"library_path" : "libEGL_nvidia.so.0"}}')
%env MUJOCO_GL=egl

import mujoco
m = mujoco.MjModel.from_xml_string("<mujoco><worldbody><geom size='0.1'/></worldbody></mujoco>")
mujoco.Renderer(m, 64, 64).render()
print("MuJoCo", mujoco.__version__, "描画OK")

上のセルで描画エラーが出る場合は、次のセルを実行して CPU 描画（OSMesa）に切り替えてください（遅くなります）。

In [ ]:
# !apt-get -qq install -y libosmesa6 > /dev/null
# %env MUJOCO_GL=osmesa
# %env PYOPENGL_PLATFORM=osmesa

In [ ]:
!python src/download_assets.py

## 2. 動作確認（真値マスクで短く実行）
シミュレーションが返す正解マスクを検出器の代わりに使い、シーン・逆運動学・清掃ループが動くことを確認します（1サイクル、低解像度）。

In [ ]:
!python run_demo.py --detector gt --max-cycles 1 --min-seconds 5 --scale 0.5 --out outputs/check_gt.mp4

In [ ]:
from IPython.display import HTML
from base64 import b64encode
def show_video(path, width=900):
    data = b64encode(open(path, "rb").read()).decode()
    return HTML(f'<video width={width} controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')
show_video("outputs/check_gt.mp4")

## 3. 合成データセット生成
切粉の個数・配置・向き、照明、台と切粉の色味、アームの映り込みをランダムに変えながら俯瞰画像を描画し、
シミュレーションの真値マスクから YOLO-seg 形式のラベルを自動作成します（手作業のアノテーション不要）。

In [ ]:
!python generate_dataset.py --n-train 400 --n-val 100

In [ ]:
# ラベルの重ね描きで確認
import cv2, glob, numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axes, sorted(glob.glob("data/chips_seg/images/train/*.jpg"))[:3]):
    img = cv2.imread(f); H, W = img.shape[:2]
    for line in open(f.replace("images", "labels").replace(".jpg", ".txt")):
        v = (np.array(line.split()[1:], float).reshape(-1, 2) * [W, H]).astype(np.int32)
        cv2.polylines(img, [v], True, (0, 0, 255), 1)
    ax.imshow(img[..., ::-1]); ax.axis("off")
plt.show()

## 4. YOLO学習・評価
`yolo11n-seg` を 1クラス（chip）で学習します。T4 で 20〜30 分程度の見込みです。

In [ ]:
!python train_yolo.py --epochs 50 --imgsz 640 --batch 16

In [ ]:
from IPython.display import Image, display
display(Image("runs/chip_seg/results.png", width=900))

In [ ]:
# 検証画像での推論結果を確認
from ultralytics import YOLO
model = YOLO("runs/chip_seg/weights/best.pt")
res = model.predict(sorted(glob.glob("data/chips_seg/images/val/*.jpg"))[:3], conf=0.4, verbose=False)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, r in zip(axes, res):
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
plt.show()

## 5. YOLOで清掃デモ → 動画出力
「加工完了（切粉が落ちる）→ YOLOで検出 → 掃く → 再検出」を繰り返します。
動画が等速で75秒以上・170秒以内になるよう、加工サイクル数を自動で調整します。

In [ ]:
!python run_demo.py --detector yolo --seed 3

In [ ]:
import json
log = json.load(open("outputs/demo_yolo_log.json", encoding="utf-8"))
print(f"動画の長さ: {log['video_seconds']:.1f} 秒 / 総ストローク数: {log['total_strokes']}")
for c in log["cycles"]:
    print(f"サイクル{c['cycle']}: 切粉 {c['n_new_chips']} 個, ストローク {c['strokes']} 回, "
          f"台上の残り(検出) {c['remaining_on_table_detected']} 個, 回収率(真値) {100*c['collected_ratio_gt']:.0f}%")
show_video("outputs/demo_yolo.mp4")

In [ ]:
# 動画と学習結果をダウンロード
!zip -q -r results.zip outputs runs/chip_seg/results.png runs/chip_seg/weights/best.pt
from google.colab import files
files.download("outputs/demo_yolo.mp4")
files.download("results.zip")

### うまくいかないときの調整ポイント（`config.yaml`）
- 検出漏れが多い → `detector.conf` を 0.3 に下げる／学習エポックを増やす
- 切粉が掃き残される → `brush.sweep_height` を下げる（例 0.001）／`planner.back_margin` を増やす
- 動画が長すぎる・短すぎる → `run_demo.py` の `--min-seconds` / `--max-seconds` を変更